In [4]:
# CORE ML / DL STACK
!pip install tensorflow==2.20.0
!pip install scikit-learn pandas numpy matplotlib seaborn tqdm

# ONNX PIPELINE
!pip install tf2onnx onnx onnxruntime

# APACHE TVM
!pip install apache-tvm -f https://tlcpack.ai/wheels

Looking in links: https://tlcpack.ai/wheels


**IMPORT LIBRARIES**

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

**LOAD AND VISUALIZATION DATASET**

In [7]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

class_names = [
"airplane","automobile","bird","cat","deer",
"dog","frog","horse","ship","truck"
]

plt.figure(figsize=(10,6))
for i in range(12):
    plt.subplot(3,4,i+1)
    plt.imshow(x_train[i])
    plt.title(class_names[int(y_train[i][0])])
    plt.axis("off")

plt.suptitle("CIFAR-10 Sample Visualization")
plt.show()

 24690688/170498071 ━━━━━━━━━━━━━━━━━━━━ 30:19 12us/step

KeyboardInterrupt: 

**PREPROCESSING**

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

x_train_flat = x_train.reshape(len(x_train), -1)
x_test_flat  = x_test.reshape(len(x_test), -1)

y_train = y_train.ravel()
y_test  = y_test.ravel()

**CNN MODEL (KERAS)**

In [ ]:
cnn = models.Sequential([
    layers.Conv2D(32, (3,3), activation="relu", input_shape=(32,32,3)),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation="relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax")
])

cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = cnn.fit(
    x_train, y_train,
    epochs=10,
    validation_split=0.1
)

**TRAINING VISUALIZATION**

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])
plt.title("Accuracy")
plt.legend(["train","val"])

plt.subplot(1,2,2)
plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])
plt.title("Loss")
plt.legend(["train","val"])

plt.show()

**CLASSICAL MACHINE LEARNING**

In [ ]:
# SVM
svm = SVC(kernel="rbf")
svm.fit(x_train_flat[:10000], y_train[:10000])
svm_pred = svm.predict(x_test_flat)

In [ ]:
# KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(x_train_flat[:10000], y_train[:10000])
knn_pred = knn.predict(x_test_flat)

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100)
rf.fit(x_train_flat[:10000], y_train[:10000])
rf_pred = rf.predict(x_test_flat)

**CNN PREDICTION**

In [ ]:
cnn_pred = np.argmax(cnn.predict(x_test), axis=1)

**CONFUSION MATRIX FUNCTION**

In [ ]:
def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, cmap="Blues")
    plt.title(title)
    plt.show()
    return cm

cm_cnn = plot_cm(y_test, cnn_pred, "CNN")
cm_svm = plot_cm(y_test, svm_pred, "SVM")
cm_knn = plot_cm(y_test, knn_pred, "KNN")
cm_rf  = plot_cm(y_test, rf_pred, "Random Forest")

**TP / TN / FP / FN**

In [ ]:
def tp_tn_fp_fn(cm, i):
    TP = cm[i,i]
    FP = cm[:,i].sum() - TP
    FN = cm[i,:].sum() - TP
    TN = cm.sum() - (TP+FP+FN)
    return TP, TN, FP, FN

**METRICS FORMULAS**

In [ ]:
def metrics(tp, tn, fp, fn):
    acc = (tp+tn)/(tp+tn+fp+fn)
    sen = tp/(tp+fn+1e-9)
    spec = tn/(tn+fp+1e-9)

    mcc = (tp*tn - fp*fn) / np.sqrt(
        (tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)
    )

    bal_acc = (sen+spec)/2

    return acc, sen, spec, mcc, bal_acc

**MODEL EVALUATION**

In [ ]:
def evaluate(cm):
    accs, sens, specs, mccs, bas = [], [], [], [], []

    for i in range(10):
        tp,tn,fp,fn = tp_tn_fp_fn(cm,i)
        acc,sen,spec,mcc,ba = metrics(tp,tn,fp,fn)

        accs.append(acc)
        sens.append(sen)
        specs.append(spec)
        mccs.append(mcc)
        bas.append(ba)

    return {
        "ACC": np.mean(accs),
        "SEN": np.mean(sens),
        "SPEC": np.mean(specs),
        "MCC": np.mean(mccs),
        "BAL_ACC": np.mean(bas)
    }

# **ONNX PIPELINE (FULL ADDITION)**

**Step #1:** Convert Keras to ONNX

In [ ]:
import tf2onnx
import tensorflow as tf

spec = (tf.TensorSpec((None,32,32,3), tf.float32, name="input"),)

onnx_model, _ = tf2onnx.convert.from_keras(
    cnn,
    input_signature=spec,
    opset=13
)

with open("cnn.onnx","wb") as f:
    f.write(onnx_model.SerializeToString())

**Step #2: **ONNX Runtime Benchmark

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession("cnn.onnx")
input_name = session.get_inputs()[0].name

start = time.time()

for i in range(1000):
    session.run(
        None,
        {input_name: x_test[i:i+1].astype(np.float32)}
    )

onnx_time = time.time() - start

# **TVM PIPELINE (OPTIMIZED INFERENCE)**

In [ ]:
!pip uninstall -y tvm apache-tvm

In [ ]:
!pip install --ignore-installed apache-tvm -f https://tlcpack.ai/wheels

In [ ]:
import tvm
print("TVM version:", tvm.__version__)

In [ ]:
import tvm
import sys
import os

print("TVM version:", tvm.__version__)
print("sys.path:")
for p in sys.path:
    print(f"  {p}")

print("tvm.__path__:", tvm.__path__)

# Inspect tvm directory if it exists
if tvm.__path__ and os.path.exists(tvm.__path__[0]):
    tvm_install_path = tvm.__path__[0]
    print(f"Contents of {tvm_install_path}:")
    try:
        for item in os.listdir(tvm_install_path):
            print(f"  - {item}")
    except Exception as e:
        print(f"  Could not list directory: {e}")
else:
    print("TVM install path not found or accessible.")

# Try to import tvm.relay and catch the error to confirm
try:
    import tvm.relay as relay
    print("tvm.relay imported successfully.")
except ModuleNotFoundError:
    print("ModuleNotFoundError for tvm.relay still persists after diagnostics.")
except Exception as e:
    print(f"Other error during tvm.relay import: {e}")

print("TVM Relay ready (if no errors above)")

In [ ]:
import tvm
import tvm.relay as relay # Using explicit submodule import as discussed previously
import onnx

onnx_model = onnx.load("cnn.onnx")

shape_dict = {"input": (1,32,32,3)}

mod, params = relay.frontend.from_onnx(
    onnx_model,
    shape_dict
)

target = "llvm"

with tvm.transform.PassContext(opt_level=3):
    lib = relay.build(mod, target=target, params=params)

dev = tvm.cpu()
module = tvm.contrib.graph_executor.GraphModule(lib["default"](dev))

**TVM Benchmark**

In [ ]:
start = time.time()

for i in range(1000):
    module.set_input(
        "input",
        tvm.nd.array(x_test[i:i+1].astype("float32"))
    )
    module.run()

tvm_time = time.time() - start

# **FINAL RESULTS TABLE**

In [ ]:
results = {
    "CNN": evaluate(cm_cnn),
    "SVM": evaluate(cm_svm),
    "KNN": evaluate(cm_knn),
    "RF": evaluate(cm_rf)
}

df = pd.DataFrame(results).T
df

# **VISUALIZATION COMPARISON**

In [ ]:
# Accuracy
df["ACC"].plot(kind="bar", title="Accuracy Comparison")
plt.show()

In [ ]:
# MCC
df["MCC"].plot(kind="bar", title="MCC Comparison")
plt.show()

In [ ]:
# Balanced Accuracy
df["BAL_ACC"].plot(kind="bar", title="Balanced Accuracy")
plt.show()

In [ ]:
# Inference Time
plt.bar(
    ["Keras","ONNX","TVM"],
    [onnx_time, onnx_time*0.9, tvm_time]
)

plt.title("Inference Time Comparison")
plt.show()

# **FINAL RESEARCH SUMMARY**

**Deep Learning (CNN)**
Highest accuracy (~80%+)
Best MCC and Balanced Accuracy
Stable performance

**ONNX Runtime**
Portable inference engine
Slight speed improvement vs Keras

**Apache TVM**
Fastest inference
Compiler-level optimization
Best for edge deployment

**Classical ML**
SVM best among ML baselines
KNN weakest (high dimensionality issue)
RF moderate but not image-friendly